# Meeting 2: The Lanczos Method

## Nandor Kovacs

### 5.3.2026

# Recall: Maximum Eigenvalue Estimation

> **Computational Problem**: Let $A \in \mathbb{H}(\mathbb{R})$ be a real psd matrix. Compute $\lambda_{\max}(A)$

> **Computational Primitive**: We can form $u \rightarrow Au$ for any vector $u \in \mathbb{R}^n$

## The Power Method
* **Input:** Input matrix $A \in \mathbb{M}_n(\mathbb{R})$, initial non-zero vector $b_0 \in \mathbb{R}^n$, number $k$ of iterations
* **Output:** Largest eigenvector estimate $b_k$, largest eigenvalue estimate $\lambda_k$

1. **function** PowerMethod($A, b_0, k$)
2. &emsp;&emsp; $b_0 = \frac{b_0}{\|b_0\|}$
3. &emsp;&emsp; **for** $i = 1, \dots, k$ **do**
4. &emsp;&emsp;&emsp;&emsp; $y_i = A b_{i-1}$
5. &emsp;&emsp;&emsp;&emsp; $b_i = \frac{y_i}{\|y_i\|}$
6. &emsp;&emsp; $\lambda_k = b_k^T A b_k$

***
### Pitfalls: 
- There is a burn in period: for around $\log(n) \cdot \frac{\lambda_1}{\lambda_1 - \lambda_2}$ steps, not much improvement is beeing made.
- If there is no spectral gap, the algorithm is pretty slow.

# Mostly Recall: The Krylov Method
**Idea:** Instead of only utilizing $A^k u$ for the eigenvector approximation, we try to make use of the information in the $k-1$ intermediary steps.

> **Def. Krylov subspace**: We call $K_{k+1}(A, \omega) := \text{span}\{\omega, A\omega, A^2 \omega, ..., A^k \omega\}$ the k'th Krylov subspace of $A$ and $\omega$. 

We will usually only write $K_{k+1}$, as it is usually clear which $A$ and $\omega$ we mean.

**Note:** $ u \in K_{k+1} \iff \exists \text{ polynomial } \phi \text{, so that } u = \phi(A)\omega$

## The Randomized Krylov Method (RKM)
We take $\omega$ randomly, and estimate the maximum eigenvalue by
$$\max{\frac{u^*Au}{||u||^2}} = \max{\frac{\omega^*A\phi^2(A)\omega}{\omega^*\phi^2(A)\omega}}$$
Where the equality is evident form $u = \phi(A)\omega$ and $A$ commuting with any polynomial of itself.

This seems like a reasonable thing to do; Previously we estimated our eigenvector in $\langle A^k \omega\rangle \subseteq K_{k+1}$. We hope that by considering the whole space $K_{k+1}$, we get a better result, as hopefully the intermediary steps contain some additional information.

## Bounds
Without a proof, we state the following bounds. Let $A \in \mathbb{H}_n(\mathbb{R})$ and $\xi_k$ be the estimated maximal eigenvalue, then:

- **with a spectral gap**: $\mathbb{E}\text{err}(\xi_k) \leq 2.589\sqrt{n} \left(1 - 2\sqrt{1 - \frac{\lambda_2}{\lambda_1}}\right)^k \leq 2.589\sqrt{n} e^{-2k\sqrt{\gamma}}$

- **without a spectral gap**: $\mathbb{E}\text{err}(\xi_k) \leq 2.575 \left\lceil \frac{\log n}{k} \right\rceil^2$


# Maximizing the Rayleigh quotient
## The Krylov scheme
1. Compute an orthonormal basis $Q_k$ for the Kylov subspace $K_k$
2. Compress $A$ to the Krylov subspace: $H_k = Q^*_kAQ_k$
3. Compute the maximal eigenvalue of the compression, $\lambda_{max}(H_k)$

# Arnold iteration
The Arnold iteration is an algorithm for computing $Q_k$. Although it has its own name, it is quite precisely just the gram-schmidt algorithm, orthogonalizing $\{\omega, A\omega, ..., A^k\omega\}$

1. **function** Arnold ($A, \omega$)
2. &emsp;&emsp; $q_1 = \frac{\omega}{||\omega||}$
3. &emsp;&emsp; **for** $i = 2, \dots, k$ **do**
4. &emsp;&emsp;&emsp;&emsp; $\hat{q}_i = Aq_{i-1} - \sum_{k=1}^{i-1}{\langle Aq_{i-1}, q_k\rangle q_k}$
5. &emsp;&emsp;&emsp;&emsp; $q_i = \frac{\hat{q_i}}{||\hat{q_i}||}$

## The algorithm in matrix form
Let $\hat{H}_k$ be the $k+1 \times k$, such that its entry at $i, j$ is equal to the coeffizients arising from the gram-schmidt algorithm in line 4 of the algorithm.

We can easily verify, that $AQ_k = Q_{k+1} \hat{H}_k$.

We note, that $\hat{H}_k$ has only entries on or above the first subdiagonal (upper Hessenberg).
<!-- $$
\frac{\hat{q_i}}{||\hat{q_i}||} = q_i \\
Aq_{i-1} - \sum_{k=1}^{i-1}{\langle Aq_{i-1}, q_k\rangle q_k} = q_i \cdot ||\hat{q_i}||\\
Aq_{i-1} = q_i \cdot ||\hat{q_i}|| + \sum_{k=1}^{i-1}{\langle Aq_{i-1}, q_k\rangle q_k}\\
\\
Aq_i = q_{i+1} \cdot ||\hat{q_{i+1}}|| + \sum_{k=1}^{i}{\langle Aq_{i}, q_k\rangle q_k}
$$ -->



# Lanczos iteration

Now we utilize our assumption from the Computational Problem earlier, that $A$ is self-adjoint, and take a look at the matrix form of the Arnold iteration. Let $H_k$ be the matrix, where we drop the bottom row of $\hat{H_k}$.
$$
\begin{align*}
AQ_k &= Q_{k+1} \hat{H}_k\\
Q_k^*AQ_k &= Q_k^*Q_{k+1} \hat{H}_k\\ 
&= [I_k | Q_k^* q_{k+1}] \hat{H}_k\\
&= [I_k | 0_v] \hat{H}_k \\
&= H_k\\
\end{align*}
$$

Therefore, we note:
- $AQ_k = Q_k H_k$
- $A$ is self adjoint $\implies H_k$ is self adjoint
- $H_k$ self adjoint, and upper Hessenberg $\implies H_k$ symmetric and tridiagonal; $H_k$ is a Jacobi matrix. 

## Algorithm speedup

This gives us a very important insight; During the algorithm, the only non zero inner products $\langle Aq_i, q_k \rangle$ are with $q_i$ and $q_{i-1}$;

$\implies Aq_{i} - \sum_{k=1}^{i}{\langle Aq_{i}, q_k\rangle q_k} = Aq_{i} - \langle Aq_{i}, q_i\rangle q_i - \langle Aq_{i}, q_{i-1}\rangle q_{i-1}$

This results in a speedup by a factor of $O(n)$.

### Note: Implementation detail
Thinking back to the Krylov scheme; We now have a way of computing $Q_k$. Note however, that as a byproduct we also get the second step for "free"; $H_k = Q_k^*AQ_k$ is the compression of $A$ to the Krylov subspace, which we recover during the gram-schmidt orthogonalization.

Recovering $\lambda_{max}(H_k)$ can be done in $O(n)$ steps via the bisection method.

# Evaluating a spectral function
## Spectral resolution
let $P_i$ be matrizes, so that:
$$
A = \sum_{1}^m{\lambda_iP_i}\\
P_iP_j = Pi\delta_{i,j}\\
\sum_{1}^m{P_i} = I\\
$$

## Spectral function
Let $A \in \mathbb{H}_n$ and $f : I \rightarrow \mathbb{R}, [\lambda_{max}(A), \lambda_{min}(A)] \subseteq I$

Then, the spectral function is defined as
$$
f(A) = \sum_{1}^m{f(\lambda_i)P_i}
$$

SVD intuition: 
$$
f(A) = Vf(\Sigma) V^\top = V\begin{bmatrix}
f(\lambda_1)  &              &        &                  & 0 \\
              & f(\lambda_2) &        &                  &   \\
              &              & \ddots &            &   \\
              &              &  & f(\lambda_{k-1}) & \\
0             &              &        &                  & f(\lambda_{k})
\end{bmatrix}V^\top
$$
Example:
- $f(x) = \frac{1}{x} \rightarrow f(A) = A^{-1}$
- $f(x) = log(x) \rightarrow f(A) = log(A)$

Evaluating spectral functions needs the whole eigenvalue decomposition in general, and is slow.

# Quadratic form $x^*f(A)x$
With access to $x^*f(A)x$, we get access to a bunch of things, including individual entries of $f(A)$; $e_i^*f(A)e_j = f(A)_{j,i}$ 

<b>Idea:</b> 
- rewrite $x^*f(A)x$ as an integral over $\mathbb{R}$.
- use estabilished numerical techniques to estimate the value of the integral

# Recall: Gaussian Quadratures


# Using Gaussian Quadratures to estimate $x^*f(A)x$
## The weighted spectral measure

# Lanczos quadrature


# Estimating Tr(f(A))
## Utility of Tr(f(A))
## Stochastic estimation

# References

1.   Joel A. Tropp, *ACM 204: Randomized Algorithms for Matrix Computations*, Caltech, CMS Lecture Notes 2020-01, Pasadena, March 2020.
2.  **Please include all other resources you have used**